# Temporal Mamba tgbl Diagnostics

This notebook is a diagnostics-oriented interface for `exp.run_temporal_tgbl`.
It keeps the same temporal sheaf + Mamba dynamics as the runner, but breaks the workflow into readable steps for:

1. dataset loading and split inspection
2. snapshot-frequency tuning
3. model and optimizer summaries
4. temporal training diagnostics
5. final validation/test results and exported CSV artifacts


In [ ]:
import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'exp').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'exp').exists(), 'Run this notebook from the repo root or notebooks/.'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from exp.parser import get_parser
import exp.run_temporal_tgbl as tgbl_runner

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)
torch.set_printoptions(sci_mode=False)


def _normalize_window_candidates(window_candidates):
    normalized = []
    seen = set()
    for window in window_candidates:
        value = None if window is None else int(window)
        if value is not None and value <= 0:
            continue
        if value in seen:
            continue
        seen.add(value)
        normalized.append(value)
    numeric = sorted(window for window in normalized if window is not None)
    return ([None] if None in seen else []) + numeric


def _make_window_candidates(timestamps, edge_ids, manual_candidates=None, limit=12):
    if manual_candidates is not None:
        return _normalize_window_candidates(manual_candidates)
    if edge_ids.numel() == 0:
        return [None]

    selected_ts = timestamps[edge_ids]
    span = int((selected_ts.max() - selected_ts.min()).item()) if selected_ts.numel() > 0 else 0
    if span <= 1:
        return [None, 1]

    raw = np.geomspace(1, span, num=limit)
    candidates = [None] + [int(x) for x in np.unique(np.round(raw).astype(int)).tolist() if int(x) > 0]
    return _normalize_window_candidates(candidates)


def _estimate_snapshot_bucket_stats(timestamps, edge_ids, time_window=None, max_edges=None):
    selected_edge_ids = edge_ids if max_edges is None else edge_ids[:max_edges]
    if selected_edge_ids.numel() == 0:
        return {
            'edge_prefix': 0,
            'raw_unique_timestamps': 0,
            'estimated_snapshots': 0,
            'estimated_mean_edges': float('nan'),
            'estimated_max_edges': 0,
            'estimated_min_edges': 0,
            'compression_ratio': float('nan'),
            'timestamp_start': None,
            'timestamp_end': None,
        }

    selected_ts = timestamps[selected_edge_ids]
    ordered_ts = selected_ts[torch.argsort(selected_ts)]
    raw_unique_timestamps = int(torch.unique_consecutive(ordered_ts).numel())

    if time_window is None:
        bucket_keys = ordered_ts
    else:
        if time_window <= 0:
            raise ValueError('time_window must be positive when provided.')
        bucket_keys = torch.div(ordered_ts - ordered_ts[0], time_window, rounding_mode='floor')

    _, bucket_counts = torch.unique_consecutive(bucket_keys, return_counts=True)
    return {
        'edge_prefix': int(selected_edge_ids.numel()),
        'raw_unique_timestamps': raw_unique_timestamps,
        'estimated_snapshots': int(bucket_counts.numel()),
        'estimated_mean_edges': float(bucket_counts.float().mean().item()),
        'estimated_max_edges': int(bucket_counts.max().item()),
        'estimated_min_edges': int(bucket_counts.min().item()),
        'compression_ratio': float(bucket_counts.numel() / raw_unique_timestamps) if raw_unique_timestamps else float('nan'),
        'timestamp_start': int(ordered_ts[0].item()),
        'timestamp_end': int(ordered_ts[-1].item()),
    }


def _estimate_time_window_grid(temporal_data, edge_ids, max_edges, candidate_windows):
    rows = []
    for time_window in candidate_windows:
        stats = _estimate_snapshot_bucket_stats(
            temporal_data.t,
            edge_ids,
            time_window=time_window,
            max_edges=max_edges,
        )
        rows.append({
            'time_window': time_window,
            'window_label': 'exact timestamps' if time_window is None else str(time_window),
            **stats,
        })
    return pd.DataFrame(rows)


def _recommend_time_window(estimate_df, target_max_snapshots=None, target_max_mean_edges=None):
    if estimate_df.empty:
        return None

    feasible = estimate_df.copy()
    if target_max_snapshots is not None:
        feasible = feasible[feasible['estimated_snapshots'] <= target_max_snapshots]
    if target_max_mean_edges is not None:
        feasible = feasible[feasible['estimated_mean_edges'] <= target_max_mean_edges]
    if feasible.empty:
        return None

    order = feasible['time_window'].map(lambda x: -1 if x is None else int(x))
    return feasible.iloc[order.argsort(kind='stable')].iloc[0].to_dict()


## 0. Hyperparameter Knobs


In [ ]:
DATASET_NAME = 'tgbl-wiki-v2'  # Example: 'tgbl-wiki-v2' or 'tgbl-review-v2'

SNAPSHOT_TIME_WINDOW = None
SNAPSHOT_TUNING_SPLIT = 'train'
SNAPSHOT_WINDOW_CANDIDATES = None
SNAPSHOT_TARGET_MAX_SNAPSHOTS = 250
SNAPSHOT_TARGET_MAX_MEAN_EDGES = 256
AUTO_PICK_SNAPSHOT_TIME_WINDOW = True

TEMPORAL_BPTT_STEPS = 8
TEMPORAL_EPOCH_PROGRESS_BAR = True
TEMPORAL_EVAL_EVERY = 2
TEMPORAL_SKIP_TRAIN_EVAL = True
TEMPORAL_EPOCHS = 50

TEMPORAL_TRAIN_EDGES = 4096
TEMPORAL_VAL_EDGES = 512
TEMPORAL_TEST_EDGES = 512

parser = get_parser()
args = parser.parse_args([
    f'--dataset={DATASET_NAME}',
    f'--temporal_dataset={DATASET_NAME}',
    '--model=TemporalMambaSheaf',
    '--stateful_temporal=True',
    '--closure_hops=1',
    f'--temporal_epochs={TEMPORAL_EPOCHS}',
    f'--temporal_train_edges={TEMPORAL_TRAIN_EDGES}',
    f'--temporal_val_edges={TEMPORAL_VAL_EDGES}',
    f'--temporal_test_edges={TEMPORAL_TEST_EDGES}',
])

args.temporal_snapshot_time_window = SNAPSHOT_TIME_WINDOW
args.temporal_bptt_steps = TEMPORAL_BPTT_STEPS
args.temporal_epoch_progress_bar = TEMPORAL_EPOCH_PROGRESS_BAR
args.temporal_eval_every = TEMPORAL_EVAL_EVERY
args.temporal_skip_train_eval = TEMPORAL_SKIP_TRAIN_EVAL

device = torch.device(f'cuda:{args.cuda}' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

config_df = pd.DataFrame([
    {'field': 'dataset_name_requested', 'value': DATASET_NAME},
    {'field': 'device', 'value': str(device)},
    {'field': 'seed', 'value': args.seed},
    {'field': 'lr', 'value': args.lr},
    {'field': 'weight_decay', 'value': args.weight_decay},
    {'field': 'temporal_epochs', 'value': args.temporal_epochs},
    {'field': 'temporal_train_edges', 'value': args.temporal_train_edges},
    {'field': 'temporal_val_edges', 'value': args.temporal_val_edges},
    {'field': 'temporal_test_edges', 'value': args.temporal_test_edges},
    {'field': 'temporal_max_edges_per_snapshot', 'value': args.temporal_max_edges_per_snapshot},
    {'field': 'temporal_snapshot_time_window', 'value': args.temporal_snapshot_time_window},
    {'field': 'temporal_bptt_steps', 'value': args.temporal_bptt_steps},
    {'field': 'temporal_epoch_progress_bar', 'value': args.temporal_epoch_progress_bar},
    {'field': 'temporal_eval_every', 'value': args.temporal_eval_every},
    {'field': 'temporal_skip_train_eval', 'value': args.temporal_skip_train_eval},
])
display(config_df)


## 1. Prepare the temporal dataset and inspect metadata


In [ ]:
requested_dataset_name = args.temporal_dataset
dataset_name, dataset, temporal_data = tgbl_runner._load_temporal_data(requested_dataset_name)
num_nodes = tgbl_runner._infer_graph_size(temporal_data)
destination_spec = tgbl_runner._destination_spec(temporal_data)
node_features = tgbl_runner._make_node_features(dataset, num_nodes).to(device)

train_edge_ids, val_edge_ids, test_edge_ids, split_source = tgbl_runner._split_edge_ids(dataset, temporal_data)
train_edge_ids = train_edge_ids[:args.temporal_train_edges]
val_edge_ids = val_edge_ids[:args.temporal_val_edges]
test_edge_ids = test_edge_ids[:args.temporal_test_edges]

split_edge_map = {
    'train': train_edge_ids,
    'val': val_edge_ids,
    'test': test_edge_ids,
}
split_edge_caps = {
    'train': args.temporal_train_edges,
    'val': args.temporal_val_edges,
    'test': args.temporal_test_edges,
}

if SNAPSHOT_TUNING_SPLIT not in split_edge_map:
    raise ValueError(f'SNAPSHOT_TUNING_SPLIT must be one of {sorted(split_edge_map)}')

candidate_windows = _make_window_candidates(
    temporal_data.t,
    split_edge_map[SNAPSHOT_TUNING_SPLIT],
    manual_candidates=SNAPSHOT_WINDOW_CANDIDATES,
)
snapshot_window_estimates = _estimate_time_window_grid(
    temporal_data,
    split_edge_map[SNAPSHOT_TUNING_SPLIT],
    split_edge_caps[SNAPSHOT_TUNING_SPLIT],
    candidate_windows,
)
snapshot_window_recommendation = _recommend_time_window(
    snapshot_window_estimates,
    target_max_snapshots=SNAPSHOT_TARGET_MAX_SNAPSHOTS,
    target_max_mean_edges=SNAPSHOT_TARGET_MAX_MEAN_EDGES,
)

display(Markdown(
    f'**Requested dataset:** `{requested_dataset_name}`  '
    f'**Loader dataset:** `{dataset_name}`  '
    f'**Split source:** `{split_source}`  '
    f'**Snapshot tuning split:** `{SNAPSHOT_TUNING_SPLIT}` with edge cap `{split_edge_caps[SNAPSHOT_TUNING_SPLIT]}`.'
))

display(snapshot_window_estimates[[
    'window_label',
    'edge_prefix',
    'estimated_snapshots',
    'estimated_mean_edges',
    'estimated_max_edges',
    'compression_ratio',
    'timestamp_start',
    'timestamp_end',
]])

if snapshot_window_recommendation is not None:
    display(Markdown(
        f"**Recommended `SNAPSHOT_TIME_WINDOW`:** `{snapshot_window_recommendation['window_label']}` "
        f"for about **{int(snapshot_window_recommendation['estimated_snapshots'])}** estimated snapshots "
        f"and **{snapshot_window_recommendation['estimated_mean_edges']:.1f}** mean edges per snapshot."
    ))

if AUTO_PICK_SNAPSHOT_TIME_WINDOW and snapshot_window_recommendation is not None:
    args.temporal_snapshot_time_window = snapshot_window_recommendation['time_window']
    display(Markdown(
        f"**Auto-pick enabled:** using `temporal_snapshot_time_window={snapshot_window_recommendation['window_label']}` for the actual snapshot build."
    ))
else:
    display(Markdown(
        f"**Using `temporal_snapshot_time_window`:** `{('exact timestamps' if args.temporal_snapshot_time_window is None else args.temporal_snapshot_time_window)}`"
    ))

train_snapshots = tgbl_runner._build_snapshots(
    temporal_data,
    train_edge_ids,
    node_features,
    max_edges=args.temporal_max_edges_per_snapshot,
    time_window=args.temporal_snapshot_time_window,
)
val_snapshots = tgbl_runner._build_snapshots(
    temporal_data,
    val_edge_ids,
    node_features,
    max_edges=args.temporal_max_edges_per_snapshot,
    time_window=args.temporal_snapshot_time_window,
)
test_snapshots = tgbl_runner._build_snapshots(
    temporal_data,
    test_edge_ids,
    node_features,
    max_edges=args.temporal_max_edges_per_snapshot,
    time_window=args.temporal_snapshot_time_window,
)

assert train_snapshots and val_snapshots, 'Temporal dataset did not produce non-empty train/validation snapshot sequences.'

dataset_metadata = {
    'dataset_requested_name': requested_dataset_name,
    'dataset_loader_name': dataset_name,
    'task_family': 'tgbl_linkprop',
    'eval_metric': getattr(dataset, 'eval_metric', 'mrr'),
    'num_nodes': num_nodes,
    'destination_offset': int(destination_spec['offset']),
    'destination_size': int(destination_spec['size']),
    'num_edges_total': int(len(temporal_data.src)),
    'train_edges_selected': int(train_edge_ids.numel()),
    'val_edges_selected': int(val_edge_ids.numel()),
    'test_edges_selected': int(test_edge_ids.numel()),
    'timestamp_min': int(temporal_data.t.min().item()),
    'timestamp_max': int(temporal_data.t.max().item()),
    'timestamp_span': int((temporal_data.t.max() - temporal_data.t.min()).item()),
    'edge_type_present': bool(getattr(temporal_data, 'edge_type', None) is not None),
    'node_feature_shape': tuple(node_features.shape),
    'node_feature_dtype': str(node_features.dtype),
    'node_feature_device': str(node_features.device),
    'snapshot_time_window': args.temporal_snapshot_time_window,
    'snapshot_grouping': 'exact timestamps' if args.temporal_snapshot_time_window is None else f'window={args.temporal_snapshot_time_window}',
}
display(pd.DataFrame(dataset_metadata.items(), columns=['field', 'value']))


In [ ]:
def summarize_snapshots(split_name, snapshots, edge_ids):
    edge_counts = [int(snapshot.src.numel()) for snapshot in snapshots]
    active_counts = [int(snapshot.active_nodes.numel()) for snapshot in snapshots]
    timestamps = [int(snapshot.timestamp.item()) for snapshot in snapshots]
    selected_ts = temporal_data.t[edge_ids]
    raw_unique_timestamps = int(torch.unique(selected_ts).numel()) if edge_ids.numel() > 0 else 0

    edge_type_values = []
    for snapshot in snapshots:
        if snapshot.edge_types is not None:
            edge_type_values.append(snapshot.edge_types)
    unique_edge_types = int(torch.unique(torch.cat(edge_type_values)).numel()) if edge_type_values else 0

    return {
        'split': split_name,
        'edge_ids_used': int(edge_ids.numel()),
        'raw_unique_timestamps': raw_unique_timestamps,
        'snapshots': len(snapshots),
        'compression_ratio': (len(snapshots) / raw_unique_timestamps) if raw_unique_timestamps else float('nan'),
        'edges_per_snapshot_mean': float(np.mean(edge_counts)) if edge_counts else float('nan'),
        'edges_per_snapshot_max': int(max(edge_counts)) if edge_counts else 0,
        'active_nodes_mean': float(np.mean(active_counts)) if active_counts else float('nan'),
        'active_nodes_max': int(max(active_counts)) if active_counts else 0,
        'timestamp_start': min(timestamps) if timestamps else None,
        'timestamp_end': max(timestamps) if timestamps else None,
        'unique_edge_types': unique_edge_types,
    }

split_summary_df = pd.DataFrame([
    summarize_snapshots('train', train_snapshots, train_edge_ids),
    summarize_snapshots('val', val_snapshots, val_edge_ids),
    summarize_snapshots('test', test_snapshots, test_edge_ids),
])
display(split_summary_df)

snapshot_preview = []
for split_name, snapshots in [('train', train_snapshots), ('val', val_snapshots), ('test', test_snapshots)]:
    for snapshot_idx, snapshot in enumerate(snapshots[:5]):
        snapshot_preview.append({
            'split': split_name,
            'snapshot_idx': snapshot_idx,
            'timestamp': int(snapshot.timestamp.item()),
            'edges': int(snapshot.src.numel()),
            'active_nodes': int(snapshot.active_nodes.numel()),
            'unique_edge_types': int(torch.unique(snapshot.edge_types).numel()) if snapshot.edge_types is not None else 0,
        })
display(pd.DataFrame(snapshot_preview))


## 2. Instantiate the model and summarize architecture + effective training config


In [ ]:
model = tgbl_runner._make_model(train_snapshots[0].edge_index, node_features, num_nodes, destination_spec, device, args)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

effective_model_args = {
    'task_family': 'tgbl_linkprop',
    'd': tgbl_runner._env_int('TGB_SHEAF_D', 2),
    'layers': tgbl_runner._env_int('TGB_LAYERS', 2),
    'hidden_channels': tgbl_runner._env_int('TGB_HIDDEN_CHANNELS', 8),
    'dropout': float(os.environ.get('TGB_DROPOUT', 0.0)),
    'temporal_d_model': args.temporal_d_model or tgbl_runner._env_int('TGB_TEMPORAL_D_MODEL', 64),
    'input_dim': int(node_features.size(1)),
    'graph_size': int(num_nodes),
    'output_dim': int(destination_spec['size']),
    'destination_offset': int(destination_spec['offset']),
    'stateful_temporal': bool(args.stateful_temporal),
    'closure_hops': int(args.closure_hops),
    'normalised': True,
    'deg_normalised': False,
    'left_weights': True,
    'right_weights': True,
    'orth': 'householder',
    'edge_weights': False,
}

model_config_df = pd.DataFrame(effective_model_args.items(), columns=['field', 'value'])
display(model_config_df)

param_rows = []
total_params = 0
trainable_params = 0
for name, param in model.named_parameters():
    count = int(param.numel())
    total_params += count
    if param.requires_grad:
        trainable_params += count
    param_rows.append({
        'name': name,
        'shape': tuple(param.shape),
        'count': count,
        'trainable': bool(param.requires_grad),
    })

param_df = pd.DataFrame(param_rows).sort_values(['count', 'name'], ascending=[False, True]).reset_index(drop=True)
display(pd.DataFrame([
    {'field': 'total_params', 'value': total_params},
    {'field': 'trainable_params', 'value': trainable_params},
    {'field': 'optimizer', 'value': type(optimizer).__name__},
    {'field': 'optimizer_lr', 'value': optimizer.param_groups[0]['lr']},
    {'field': 'optimizer_weight_decay', 'value': optimizer.param_groups[0]['weight_decay']},
]))
display(param_df.head(15))


## 3. Train the model and log train/validation/test behavior


In [ ]:
metric_name = getattr(dataset, 'eval_metric', 'mrr')
history = []
best_val_metric = float('-inf')
best_test_metric = float('nan')
best_epoch = -1
best_state_dict = None

start = time.perf_counter()
for epoch in tqdm(range(args.temporal_epochs), desc='Temporal tgbl training'):
    model.reset_temporal_state()
    train_loss = tgbl_runner._run_epoch(
        model,
        optimizer,
        train_snapshots,
        destination_spec,
        bptt_steps=args.temporal_bptt_steps,
        show_progress=args.temporal_epoch_progress_bar,
        epoch_label=f'Epoch {epoch + 1}/{args.temporal_epochs}',
    )

    should_eval = ((epoch + 1) % max(args.temporal_eval_every, 1) == 0) or (epoch == args.temporal_epochs - 1)
    train_metric = float('nan')
    val_metric = float('nan')
    test_metric = float('nan')
    train_eval_loss = float('nan')
    val_eval_loss = float('nan')
    test_eval_loss = float('nan')
    train_state_memory_norm = float('nan')
    val_state_memory_norm = float('nan')
    test_state_memory_norm = float('nan')

    if should_eval:
        if args.temporal_skip_train_eval:
            train_state = tgbl_runner._advance_context(model, train_snapshots)
        else:
            train_metric, train_eval_loss, train_state = tgbl_runner._evaluate_model_streaming(
                dataset_name,
                dataset,
                train_snapshots,
                model,
                destination_spec,
                initial_state=None,
                split_mode='train',
                compute_metric=False,
                compute_loss=True,
            )

        val_metric, val_eval_loss, val_state = tgbl_runner._evaluate_model_streaming(
            dataset_name,
            dataset,
            val_snapshots,
            model,
            destination_spec,
            initial_state=train_state,
            split_mode='val',
        )
        test_metric, test_eval_loss, test_state = tgbl_runner._evaluate_model_streaming(
            dataset_name,
            dataset,
            test_snapshots,
            model,
            destination_spec,
            initial_state=val_state,
            split_mode='test',
        )

        train_state_memory_norm = float(train_state.memory.norm().detach().cpu()) if train_state is not None and train_state.memory is not None else float('nan')
        val_state_memory_norm = float(val_state.memory.norm().detach().cpu()) if val_state is not None and val_state.memory is not None else float('nan')
        test_state_memory_norm = float(test_state.memory.norm().detach().cpu()) if test_state is not None and test_state.memory is not None else float('nan')

        if np.isfinite(val_metric) and val_metric > best_val_metric:
            best_val_metric = float(val_metric)
            best_test_metric = float(test_metric)
            best_epoch = epoch
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'evaluated': bool(should_eval),
        f'train_{metric_name}': float(train_metric),
        f'val_{metric_name}': float(val_metric),
        f'test_{metric_name}': float(test_metric),
        'train_eval_loss': train_eval_loss,
        'val_eval_loss': val_eval_loss,
        'test_eval_loss': test_eval_loss,
        'train_state_memory_norm': train_state_memory_norm,
        'val_state_memory_norm': val_state_memory_norm,
        'test_state_memory_norm': test_state_memory_norm,
    })

elapsed = time.perf_counter() - start
history_df = pd.DataFrame(history)

summary_df = pd.DataFrame([
    {'field': f'best_val_{metric_name}', 'value': best_val_metric},
    {'field': f'best_test_{metric_name}', 'value': best_test_metric},
    {'field': 'best_epoch', 'value': best_epoch},
    {'field': 'elapsed_sec', 'value': elapsed},
    {'field': 'epochs_completed', 'value': len(history_df)},
])
display(summary_df)
display(history_df.tail(10))


In [ ]:
eval_history_df = history_df[history_df['evaluated']] if 'evaluated' in history_df else history_df.copy()


def _plot_available(ax, df, columns, title):
    available = [col for col in columns if col in df.columns and not df[col].isna().all()]
    if available:
        df.plot(x='epoch', y=available, ax=ax, title=title)
    else:
        ax.set_title(title)
        ax.text(0.5, 0.5, 'No evaluated data to plot yet', ha='center', va='center', transform=ax.transAxes)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

history_df.plot(x='epoch', y='train_loss', ax=axes[0, 0], title='Training loss')
_plot_available(axes[0, 1], eval_history_df, [f'train_{metric_name}', f'val_{metric_name}', f'test_{metric_name}'], f'{metric_name} over evaluated epochs')
_plot_available(axes[1, 0], eval_history_df, ['train_eval_loss', 'val_eval_loss', 'test_eval_loss'], 'Evaluation loss over evaluated epochs')
_plot_available(axes[1, 1], eval_history_df, ['train_state_memory_norm', 'val_state_memory_norm', 'test_state_memory_norm'], 'Temporal memory norm')

for ax in axes.flat:
    ax.set_xlabel('epoch')

plt.tight_layout()
plt.show()


## 4. Final diagnostic summary


In [ ]:
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

model.reset_temporal_state()
if args.temporal_skip_train_eval:
    best_train_metric = float('nan')
    best_train_loss = float('nan')
    best_train_state = tgbl_runner._advance_context(model, train_snapshots)
else:
    best_train_metric, best_train_loss, best_train_state = tgbl_runner._evaluate_model_streaming(
        dataset_name,
        dataset,
        train_snapshots,
        model,
        destination_spec,
        initial_state=None,
        split_mode='train',
        compute_metric=False,
        compute_loss=True,
    )

best_val_metric_stream, best_val_loss_stream, best_val_state = tgbl_runner._evaluate_model_streaming(
    dataset_name,
    dataset,
    val_snapshots,
    model,
    destination_spec,
    initial_state=best_train_state,
    split_mode='val',
)
best_test_metric_stream, best_test_loss_stream, best_test_state = tgbl_runner._evaluate_model_streaming(
    dataset_name,
    dataset,
    test_snapshots,
    model,
    destination_spec,
    initial_state=best_val_state,
    split_mode='test',
)

final_diagnostics = pd.DataFrame([
    {
        'split': 'train',
        metric_name: best_train_metric,
        'loss': best_train_loss,
        'num_snapshots': len(train_snapshots),
        'state_memory_norm': float(best_train_state.memory.norm().detach().cpu()) if best_train_state and best_train_state.memory is not None else float('nan'),
    },
    {
        'split': 'val',
        metric_name: best_val_metric_stream,
        'loss': best_val_loss_stream,
        'num_snapshots': len(val_snapshots),
        'state_memory_norm': float(best_val_state.memory.norm().detach().cpu()) if best_val_state and best_val_state.memory is not None else float('nan'),
    },
    {
        'split': 'test',
        metric_name: best_test_metric_stream,
        'loss': best_test_loss_stream,
        'num_snapshots': len(test_snapshots),
        'state_memory_norm': float(best_test_state.memory.norm().detach().cpu()) if best_test_state and best_test_state.memory is not None else float('nan'),
    },
])
display(final_diagnostics)

display(Markdown(
    f'Best validation `{metric_name}`: **{best_val_metric:.4f}** at epoch **{best_epoch}**  '
    f'Best corresponding test `{metric_name}`: **{best_test_metric:.4f}**  '
    f'Total runtime: **{elapsed:.2f}s**'
))


In [ ]:
results_dir = REPO_ROOT / 'results' / f'{dataset_name}_{args.model}_{int(time.time())}'
results_dir.mkdir(parents=True, exist_ok=True)
config_df.to_csv(results_dir / 'config.csv', index=False)
model_config_df.to_csv(results_dir / 'model_config.csv', index=False)
split_summary_df.to_csv(results_dir / 'split_summary.csv', index=False)
snapshot_window_estimates.to_csv(results_dir / 'snapshot_window_estimates.csv', index=False)
summary_df.to_csv(results_dir / 'training_summary.csv', index=False)
history_df.to_csv(results_dir / 'history.csv', index=False)
final_diagnostics.to_csv(results_dir / 'final_diagnostics.csv', index=False)
print(f'Saved diagnostics to {results_dir}')
